# **XLMRoBERTa مدل **

In [ ]:
from transformers import AutoTokenizer
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoModel
from google.colab import drive
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import AdamW, get_linear_schedule_with_warmup
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

**استفاده از کل داده‌ها**

In [ ]:
train_data = pd.read_csv('/content/drive/MyDrive/Poem Meter Dataset/train_samples.csv')
validation_data = pd.read_csv('/content/drive/MyDrive/Poem Meter Dataset/validation_samples.csv')
test_data = pd.read_csv('/content/drive/MyDrive/Poem Meter Dataset/test_samples.csv')

print(f"\nNumber of samples in Train Data: {len(train_data)}")
print(f"Number of samples in Validation Data: {len(validation_data)}")
print(f"Number of samples in Test Data: {len(test_data)}")



Number of samples in Train Data: 749184
Number of samples in Validation Data: 42545
Number of samples in Test Data: 42545


In [ ]:
train_data.head()

,poet_id,poem_id,v_order,poem_text,metre,metre_2
0,9,37309,929,شرح بسیاری بگفت از کائنات,فاعلاتن فاعلاتن فاعلن,-U---U---U-
1,34,62218,14,چه جای رفتن باغ است و گشتن بستان,مفاعلن فعلاتن مفاعلن فعلن,U-U-UU--U-U-UU-
2,9,37296,72,پای خود آرم برون وبر پرم,فاعلاتن فاعلاتن فاعلن,-U---U---U-
3,6,9174,183,چو گوهر برآمود زنگی به تاج,فعولن فعولن فعولن فعل,U--U--U--U-
4,5,8021,28,هست جاری دجله‌ای همچون شکر,فاعلاتن فاعلاتن فاعلن,-U---U---U-


In [ ]:
validation_data.head()

,poet_id,poem_id,v_order,poem_text,metre,metre_2
0,5,4063,38,که من با چو و با تو را نمی‌دانم نمی‌دانم,مفاعیلن مفاعیلن مفاعیلن مفاعیلن,U---U---U---U---
1,5,8151,56,نیست آن جز حیلهٔ نفس لیم,فاعلاتن فاعلاتن فاعلن,-U---U---U-
2,13,13975,9,پراپرند زطمع بازو، جغدکان بی‌رنج,مفاعلن فعلاتن مفاعلن فعلن,U-U-UU--U-U-UU-
3,24,66096,66,ولی شد چار دای از چار یارش,مفاعیلن مفاعیلن فعولن,U---U---U--
4,9,33851,88,همی جفتی طلب چون چشم آخر!,مفاعیلن مفاعیلن فعولن,U---U---U--


**حذف ستون‌های غیر ضروری**

In [ ]:
train_data = train_data.drop(columns=['poet_id', 'poem_id', 'v_order', 'metre_2'])
validation_data = validation_data.drop(columns=['poet_id', 'poem_id', 'v_order', 'metre_2'])
test_data = test_data.drop(columns=['poet_id', 'poem_id', 'v_order'])
train_data

,poem_text,metre
0,شرح بسیاری بگفت از کائنات,فاعلاتن فاعلاتن فاعلن
1,چه جای رفتن باغ است و گشتن بستان,مفاعلن فعلاتن مفاعلن فعلن
2,پای خود آرم برون وبر پرم,فاعلاتن فاعلاتن فاعلن
3,چو گوهر برآمود زنگی به تاج,فعولن فعولن فعولن فعل
4,هست جاری دجله‌ای همچون شکر,فاعلاتن فاعلاتن فاعلن
...,...,...
749179,ز درش به روز من ار چه دور همی روم,متفاعلن متفاعلن متفاعلن متفاعلن
749180,رخ او گلفشان شود نظرم گلستان شود,فعلاتن مفاعلن فعلاتن مفاعلن
749181,بسر تو کین دل‌خسته را به نسیم خود خبری کنی,متفاعلن متفاعلن متفاعلن متفاعلن
749182,آزاده نژاد از درم خرید,مفعول مفاعیل فاعلن


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


**آماده‌سازی مصرع و وزن برای ورود به مدل XLMRoBERTa**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
token_lengths = [len(tokenizer.encode(text, truncation=False)) for text in train_data['poem_text'].tolist()]
average_length = sum(token_lengths) / len(token_lengths)

print(f"Average token length: {average_length}")
print(f"Max token length in dataset: {max(token_lengths)}")
print(f"Suggested max sequence length (covering ~90%): {int(average_length * 1.5)}")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Average token length: 11.03515424782163
Max token length in dataset: 35
Suggested max sequence length (covering ~90%): 16


**توکنایز مصرع‌ها**

In [ ]:
max_seq_length = 16
train_encodings = tokenizer(
    train_data['poem_text'].tolist(),
    truncation=True,
    padding=True,
    max_length=max_seq_length,
    return_tensors="pt"
)

validation_encodings = tokenizer(
    validation_data['poem_text'].tolist(),
    truncation=True,
    padding=True,
    max_length=max_seq_length,
    return_tensors="pt"
)

test_encodings = tokenizer(
    test_data['poem_text'].tolist(),
    truncation=True,
    padding=True,
    max_length=max_seq_length,
    return_tensors="pt"
)

print("Train Encodings (input_ids) Shape:", train_encodings['input_ids'].shape)
print("Validation Encodings (input_ids) Shape:", validation_encodings['input_ids'].shape)
print("Test Encodings (input_ids) Shape:", test_encodings['input_ids'].shape)


Train Encodings (input_ids) Shape: torch.Size([749184, 16])
Validation Encodings (input_ids) Shape: torch.Size([42545, 16])
Test Encodings (input_ids) Shape: torch.Size([42545, 16])


**تبدیل برچسب‌های وزن به مقدار عددی یکتا**

In [ ]:
label_encoder = LabelEncoder()

train_labels = label_encoder.fit_transform(train_data['metre'])
validation_labels = label_encoder.transform(validation_data['metre'])

train_labels_tensor = torch.tensor(train_labels)
validation_labels_tensor = torch.tensor(validation_labels)

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

print("Train Labels Shape:", train_labels_tensor.shape)
print("Validation Labels Shape:", validation_labels_tensor.shape)


Train Labels Shape: torch.Size([749184])
Validation Labels Shape: torch.Size([42545])


In [ ]:
label_mapping

{'فاعلات فع فاعلات فع': 0,
 'فاعلاتن فاعلاتن فاعلاتن': 1,
 'فاعلاتن فاعلاتن فاعلاتن فاعلاتن': 2,
 'فاعلاتن فاعلاتن فاعلاتن فاعلن': 3,
 'فاعلاتن فاعلاتن فاعلن': 4,
 'فاعلن مفاعیلن فاعلن مفاعیلن': 5,
 'فعلات فاعلاتن فعلات فاعلاتن': 6,
 'فعلاتن فعلاتن فعلاتن فع': 7,
 'فعلاتن فعلاتن فعلاتن فعلاتن': 8,
 'فعلاتن فعلاتن فعلاتن فعلن': 9,
 'فعلاتن فعلاتن فعلن': 10,
 'فعلاتن مفاعلن فعلاتن': 11,
 'فعلاتن مفاعلن فعلاتن مفاعلن': 12,
 'فعلاتن مفاعلن فعلن': 13,
 'فعولن فعولن فعولن فعل': 14,
 'فعولن فعولن فعولن فعولن': 15,
 'فعولن فعولن مفاعلن': 16,
 'فعولن مفاعلن فعولن مفاعلن': 17,
 'متفاعلن متفاعلن متفاعلن متفاعلن': 18,
 'مستفعلن فع مستفعلن فع': 19,
 'مستفعلن فعلن مستفعلن فعلن': 20,
 'مستفعلن مستفعلن مستفعلن': 21,
 'مستفعلن مستفعلن مستفعلن مستفعلن': 22,
 'مفاعلن فع مفاعلن فع': 23,
 'مفاعلن فعلاتن مفاعلن فعلاتن': 24,
 'مفاعلن فعلاتن مفاعلن فعلن': 25,
 'مفاعیل مفاعیل مفاعیل فعولن': 26,
 'مفاعیلن مفاعیلن فعولن': 27,
 'مفاعیلن مفاعیلن مفاعیلن': 28,
 'مفاعیلن مفاعیلن مفاعیلن مفاعیلن': 29,
 'مفتعلن فاعلات

In [ ]:
train_dataset = TensorDataset(
    train_encodings['input_ids'],
    train_encodings['attention_mask'],
    train_labels_tensor
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

validation_dataset = TensorDataset(
    validation_encodings['input_ids'],
    validation_encodings['attention_mask'],
    validation_labels_tensor
)
validation_loader = DataLoader(validation_dataset, batch_size=32)

test_dataset = TensorDataset(
    test_encodings['input_ids'],
    test_encodings['attention_mask']
)
test_loader = DataLoader(test_dataset, batch_size=32)


**XLMRoBERTa مدل**

In [ ]:
num_classes = len(label_mapping)
class XLMRobertaClassifier(nn.Module):
    def __init__(self, num_classes):
        super(XLMRobertaClassifier, self).__init__()
        self.roberta = AutoModel.from_pretrained("xlm-roberta-base")

        # Classification head
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):

        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)

        # Get the hidden states of the [CLS] token (first token)
        cls_output = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(cls_output)

        return logits


model = XLMRobertaClassifier(num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(model)


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

XLMRobertaClassifier(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
   

**آموزش و ارزیابی**

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

num_epochs = 3

total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
def train(model, train_loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc="Training"):

        input_ids, attention_mask, labels = [item.to(device) for item in batch]

        # Forward pass
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Compute loss
        loss = criterion(outputs, labels)
        total_loss += loss.item()

        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)
    return avg_loss


In [ ]:
def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):

            input_ids, attention_mask, labels = [item.to(device) for item in batch]

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            # Compute loss
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Get predictions
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(val_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=1)
    recall = recall_score(all_labels, all_preds, average="macro", zero_division=1)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=1)

    return avg_loss, accuracy, precision, recall, f1


In [ ]:
best_f1 = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss = train(model, train_loader, optimizer, scheduler, criterion, device)
    print(f"Training Loss: {train_loss:.4f}")

    val_loss, val_accuracy, val_precision, val_recall, val_f1 = evaluate(model, validation_loader, criterion, device)
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1 Score: {val_f1:.4f}")

    torch.save(model.state_dict(), f"xlmroberta_model_epoch_{epoch+1}.pth")
    print(f"Model saved for epoch {epoch + 1} with F1 Score: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_xlmroberta_model.pth")
        print(f"New best model saved with F1 Score: {val_f1:.4f}")

print("Training complete!")



Epoch 1/3


Training: 100%|██████████| 23412/23412 [1:01:06<00:00,  6.39it/s]


Training Loss: 0.5972


Evaluating: 100%|██████████| 1330/1330 [00:38<00:00, 34.59it/s]


Validation Loss: 0.2349, Accuracy: 0.9381, Precision: 0.8835, Recall: 0.5620, F1 Score: 0.5629
Model saved for epoch 1 with F1 Score: 0.5629
New best model saved with F1 Score: 0.5629

Epoch 2/3


...

Validation Loss: 0.1386, Accuracy: 0.9622, Precision: 0.9258, Recall: 0.6597, F1 Score: 0.6748
Model saved for epoch 2 with F1 Score: 0.6748
New best model saved with F1 Score: 0.6748

Epoch 3/3


Training: 100%|██████████| 23412/23412 [1:01:05<00:00,  6.39it/s]


Training Loss: 0.0821


Evaluating: 100%|██████████| 1330/1330 [00:38<00:00, 34.59it/s]


Validation Loss: 0.1124, Accuracy: 0.9703, Precision: 0.9333, Recall: 0.7104, F1 Score: 0.7213
Model saved for epoch 3 with F1 Score: 0.7213
New best model saved with F1 Score: 0.7213
Training complete!


**پیش بینی وزن داده‌های تست**

In [ ]:
model.load_state_dict(torch.load("best_xlmroberta_model.pth"))
model.to(device)
model.eval()

def predict(model, test_loader, device):
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting on Test Data"):

            input_ids, attention_mask = [item.to(device) for item in batch]

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            # Get predictions
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())

    return all_preds


from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/test_predictions.csv'
test_predictions = predict(model, test_loader, device)
predicted_labels = [label_encoder.inverse_transform([pred])[0] for pred in test_predictions]

test_data['predicted_label'] = predicted_labels
test_data[['poem_text', 'predicted_label']].to_csv(save_path, index=False)

print(test_data[['poem_text', 'predicted_label']].head(10))



<ipython-input-18-aaf9ef476351>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_xlmroberta_model.pth"))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Predicting on Test Data: 100%|██████████| 1330/1330 [00:54<00:00, 24.22it/s]


                               poem_text                predicted_label
0       گر درخور عشق آید خرم چو دمشق آید    مفعول مفاعیلن مفعول مفاعیلن
1         ای صدر جهان جهان ندارد چو تویی        مفعول مفاعیل مفاعیل فعل
2               شوی بی‌گزند از بد بدگمان          فعولن فعولن فعولن فعل
3  گه‌ کلیمی سازد از موسی و در دستش‌ کند  فاعلاتن فاعلاتن فاعلاتن فاعلن
4                بسی گفتی و در آخر رسیدی          مفاعیلن مفاعیلن فعولن
5            پس ز کوزه آن تلابد که دروست          فاعلاتن فاعلاتن فاعلن
6             بگفتند آنچه او را رونمودست          مفاعیلن مفاعیلن فعولن
7                 بی‌آزاری زیردستان گزین          فعولن فعولن فعولن فعل
8    خصم بد عهدت که کهف ملک را هشتم کسست  فاعلاتن فاعلاتن فاعلاتن فاعلن
9              واقف اسرار آن جانان نه‌ای          فاعلاتن فاعلاتن فاعلن


In [ ]:
test_data[['poem_text', 'predicted_label']].head(10)

,poem_text,predicted_label
0,گر درخور عشق آید خرم چو دمشق آید,مفعول مفاعیلن مفعول مفاعیلن
1,ای صدر جهان جهان ندارد چو تویی,مفعول مفاعیل مفاعیل فعل
2,شوی بی‌گزند از بد بدگمان,فعولن فعولن فعولن فعل
3,گه‌ کلیمی سازد از موسی و در دستش‌ کند,فاعلاتن فاعلاتن فاعلاتن فاعلن
4,بسی گفتی و در آخر رسیدی,مفاعیلن مفاعیلن فعولن
5,پس ز کوزه آن تلابد که دروست,فاعلاتن فاعلاتن فاعلن
6,بگفتند آنچه او را رونمودست,مفاعیلن مفاعیلن فعولن
7,بی‌آزاری زیردستان گزین,فعولن فعولن فعولن فعل
8,خصم بد عهدت که کهف ملک را هشتم کسست,فاعلاتن فاعلاتن فاعلاتن فاعلن
9,واقف اسرار آن جانان نه‌ای,فاعلاتن فاعلاتن فاعلن
